In [ ]:
import torch
import numpy as np

from utils import *
from src.iipw import *


d1 = 1000
d2 = 100
r = 10
p = 0.1
ob = 50
sample = "iid"
if torch.cuda.is_available():
    device = 'cuda:1'
else:
    device = 'cpu'

# dataset
dataset = "syn"
err_T_list = []
err_T_p_list = []
M_mean_list = []
M_var_list = []
M = load_normalized_data_syn(r, d1, d2, device)
MTM = M.T @ M
runs = 100000

print(f"d1: {d1}, d2: {d2}, r: {r}, p: {p}, ob: {ob}, sample: {sample}, runs: {runs}")

A_list = []
B_list = []
T_list = []
T_err_list = []
for run in tqdm(range(runs)):

    if sample == "iid":
        # IID sample
        observed_M, masks = get_uniform_masks(M, p)
    else:
        # few entry sample
        observed_M, masks = get_random_samples_per_row(M.cpu().numpy(), ob)
        p = ob/d2
        observed_M = torch.from_numpy(observed_M).float().to(device)
        masks = torch.from_numpy(masks).to(device)
    
    
    normalized_MTM = MTM / d1
    A =  observed_M.T @ observed_M

    B = (1 * (observed_M != 0)).float().T @ (1 * (observed_M != 0).float())
    B = B + (B == 0) * 1
    B_list.append(B.unsqueeze(0).cpu())

    A_list.append(A.unsqueeze(0).cpu())
    #T = iipw_T(observed_M)
    T = A / B
    T_list.append(T.unsqueeze(0).cpu())
    T_err_list.append(compute_err_tensor(estimation_matrix=T, groundtruth_matrix=normalized_MTM).item())

diag_mask = ~torch.eye(T.size(0), dtype=bool, device='cpu')
# statistics of A
A_mean, A_var = compute_mean_and_var(A_list)
A_mean = A_mean.to(device)
A_var = A_var.to(device)
print(A_var.shape)
A_mean_diag = torch.diag(A_mean)
A_mean_off_diag = A_mean[diag_mask]
A_var_diag = torch.diag(A_var)
A_var_off_diag = A_var[diag_mask]

# statistics of B
B_mean, B_var = compute_mean_and_var(B_list)
B_mean = B_mean.to(device)
B_var = B_var.to(device)
B_mean_diag = torch.diag(B_mean)
B_mean_off_diag = B_mean[diag_mask]
B_var_diag = torch.diag(B_var)
B_var_off_diag = B_var[diag_mask]

# statistics of T
T_mean, T_var = compute_mean_and_var(T_list)
T_mean = T_mean.to(device)
T_var = T_var.to(device)
T_mean_diag = torch.diag(T_mean)
T_mean_off_diag = T_mean[diag_mask]
T_var_diag = torch.diag(T_var)
T_var_off_diag = T_var[diag_mask]

# statistics of MTM
MTM_diag = torch.diag(MTM)
MTM_off_diag = MTM[diag_mask]
MTM_diag_mean = MTM_diag.mean()
MTM_off_diag_mean = MTM_off_diag.mean()

# check
print("avg A diag: ", A_mean_diag.mean())
print("avg A off diag: ", A_mean_off_diag.mean())
print("avg B diag: ", B_mean_diag.mean())
print("avg B off diag: ", B_mean_off_diag.mean())
print("avg T diag: ", T_mean_diag.mean())
print("avg T off diag: ", T_mean_off_diag.mean())
print("avg MTM diag: ", MTM_diag_mean)
print("avg MTM off diag: ", MTM_off_diag_mean)

cov_AB = (p**2-p**4)*MTM

# Compute the estimate
item1 = A_var / B_mean**2
item2 = A_mean**2 * B_var / B_mean**4
item3 = 2 * A_mean * cov_AB / B_mean**3

estimation_eq7 = item1 + item2 - item3
print("avg Estimation of eq7: ", estimation_eq7.mean())
print("avg Var of T: ", T_var.mean())

#print(MTM_off_diag)
#print(MTM_diag)
estimate_offdiag = A_var_off_diag / (d1**2 * p**4) - ((1-p**2) * (MTM_off_diag**2)) / (d1**3 * p**2)
estimate_diag = A_var_diag / (d1**2 * p**2) - ((1-p) * (MTM_diag**2)) / (d1**3 * p)


print("estimate diag", estimate_diag.mean())
print("estimate off diag", estimate_offdiag.mean())
print("var T diag", T_var_diag.mean())
print("var T off diag", T_var_off_diag.mean())



